# 09 — INbreast Zero-Shot Evaluation (2-Model Ensemble)

Evaluates the **2-model ensemble** (DaViT-CC + DaViT-MLO only) on the INbreast dataset.

**Critical note:** DaViT-ALL is explicitly excluded here because it was trained on INbreast data. Including it would constitute data leakage. The `predict_inbreast()` method enforces this automatically.

Expected result (paper): Accuracy = 96.59%, Balanced Accuracy = 95.35%

In [ ]:
import torch
from torch.utils.data import DataLoader

from multiview_davit.config import load_config
from multiview_davit.models.ensemble import MultiViewEnsemble
from multiview_davit.data.datasets import PairedViewDataset
from multiview_davit.data.transforms import build_inference_transform
from multiview_davit.evaluation.ensemble_eval import evaluate_ensemble, dump_predictions_csv
from multiview_davit.evaluation.confusion import plot_confusion_matrix
from multiview_davit.evaluation.stats import wilson_ci
from sklearn.metrics import confusion_matrix

ens_cfg  = load_config('configs/ensemble.yaml')
data_cfg = load_config('configs/data.yaml')
device   = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Load ensemble — DaViT-ALL checkpoints are loaded but will NOT be used for INbreast
ensemble = MultiViewEnsemble.from_config(ens_cfg, device=device)

inbreast_ds = PairedViewDataset(
    data_cfg.inbreast.csv_path,
    transform=build_inference_transform(ens_cfg.inference.input_size)
)
inbreast_loader = DataLoader(inbreast_ds, batch_size=ens_cfg.inference.batch_size,
                              shuffle=False, num_workers=4)
print(f'INbreast pairs: {len(inbreast_ds)}')

In [ ]:
# mode='inbreast' → calls predict_inbreast() → 2-model only (DaViT-ALL excluded)
metrics = evaluate_ensemble(ensemble, inbreast_loader, device=device, mode='inbreast')

n = len(inbreast_ds)
n_correct = int(round(metrics['accuracy'] * n))
lo, hi = wilson_ci(n_correct, n)

print(f"Accuracy:          {metrics['accuracy']*100:.2f}%  (95% CI: [{lo*100:.2f}%, {hi*100:.2f}%])")
print(f"Balanced Accuracy: {metrics['balanced_accuracy']*100:.2f}%")
print(f"F1 (weighted):     {metrics['f1']:.4f}")
print(f"AUC:               {metrics['auc']:.4f}")
print(f"Sensitivity:       {metrics['sensitivity']:.4f}")
print(f"Specificity:       {metrics['specificity']:.4f}")

In [ ]:
cm = confusion_matrix(metrics['y_true'], metrics['y_pred'])
fig = plot_confusion_matrix(cm, save_path='results/inbreast_confusion.png')

dump_predictions_csv(metrics['y_true'], metrics['y_pred'], metrics['y_prob'],
                     save_path='results/inbreast_predictions.csv')